In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import RegexpTokenizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

EMBEDDING_DIM = 64
HIDDEN_DIM = 128
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
TEST_SIZE = 0.2
RANDOM_STATE = 42
DATASET_PATH = "dataset/history_pre_clean"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
tokenizer = RegexpTokenizer(r"\w+|[^\w\s]")

def clean_command(cmd: str) -> list[str]:
    return tokenizer.tokenize(cmd.lower())

with open(DATASET_PATH, "r") as f:
    commands = f.readlines()

tokenized_commands = [clean_command(cmd) for cmd in commands]

all_tokens = [token for cmd_tokens in tokenized_commands for token in cmd_tokens]

vocab: dict[str, int] = {word: i + 2 for i, word in enumerate(sorted(set(all_tokens)))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1
inv_vocab: dict[int, str] = {i: word for word, i in vocab.items()}
VOCAB_SIZE = len(vocab)

In [4]:
input_sequences: list[list[int]] = []
for tokens in tokenized_commands:
    encoded = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    if len(encoded) < 2:
        continue
    for n in range(2, len(encoded) + 1):
        input_sequences.append(encoded[:n])

max_len = max(len(seq) for seq in input_sequences)
context_len = max_len - 1

context_padded = [[0] * (context_len - len(seq[:-1])) + seq[:-1] for seq in input_sequences]
targets = [seq[-1] for seq in input_sequences]

X = torch.tensor(context_padded, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)
print(f"Dataset shape: X={X.shape}, y={y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

Dataset shape: X=torch.Size([11256, 61]), y=torch.Size([11256])


In [5]:
class CommandsPrediction(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        logits = self.fc(out[:, -1, :])
        return logits


In [6]:
model = CommandsPrediction(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)
    print(f"Epoch {epoch + 1:3d} | Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']}")

Epoch   1 | Loss: 5.2483 | LR: 0.001
Epoch   2 | Loss: 4.4062 | LR: 0.001
Epoch   3 | Loss: 3.7938 | LR: 0.001
Epoch   4 | Loss: 3.3820 | LR: 0.001
Epoch   5 | Loss: 3.0905 | LR: 0.001
Epoch   6 | Loss: 2.8608 | LR: 0.001
Epoch   7 | Loss: 2.6688 | LR: 0.001
Epoch   8 | Loss: 2.5143 | LR: 0.001
Epoch   9 | Loss: 2.3825 | LR: 0.001
Epoch  10 | Loss: 2.2797 | LR: 0.001
Epoch  11 | Loss: 2.1839 | LR: 0.001
Epoch  12 | Loss: 2.1043 | LR: 0.001
Epoch  13 | Loss: 2.0359 | LR: 0.001
Epoch  14 | Loss: 1.9774 | LR: 0.001
Epoch  15 | Loss: 1.9266 | LR: 0.001
Epoch  16 | Loss: 1.8803 | LR: 0.001
Epoch  17 | Loss: 1.8396 | LR: 0.001
Epoch  18 | Loss: 1.8083 | LR: 0.001
Epoch  19 | Loss: 1.7739 | LR: 0.001
Epoch  20 | Loss: 1.7499 | LR: 0.001
Epoch  21 | Loss: 1.7241 | LR: 0.001
Epoch  22 | Loss: 1.7044 | LR: 0.001
Epoch  23 | Loss: 1.6837 | LR: 0.001
Epoch  24 | Loss: 1.6681 | LR: 0.001
Epoch  25 | Loss: 1.6556 | LR: 0.001
Epoch  26 | Loss: 1.6381 | LR: 0.001
Epoch  27 | Loss: 1.6316 | LR: 0.001
E

In [7]:
def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            total_loss += criterion(logits, labels).item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels_np = np.array(all_labels)
    all_preds_np = np.array(all_preds)

    avg_loss = total_loss / len(test_loader)
    perplexity = np.exp(avg_loss)
    acc = accuracy_score(all_labels_np, all_preds_np)
    f1_w = f1_score(all_labels_np, all_preds_np, average="weighted", zero_division=0)
    f1_m = f1_score(all_labels_np, all_preds_np, average="macro", zero_division=0)

    print("─" * 40)

    unique_labels = np.unique(all_labels_np)
    target_names = [inv_vocab[i] for i in unique_labels]
    print("\nClassification report (per token):")
    print(
        classification_report(
            all_labels_np,
            all_preds_np,
            labels=unique_labels,
            target_names=target_names,
            zero_division=0,
        )
    )
    print("─" * 40)
    print(f"Accuracy:          {acc:.4%}")
    print(f"Perplexity:        {perplexity:.4f}")
    print(f"F1 (weighted):     {f1_w:.4f}")
    print(f"F1 (macro):        {f1_m:.4f}")

evaluate()

────────────────────────────────────────

Classification report (per token):
                      precision    recall  f1-score   support

                   "       0.87      0.82      0.84        33
                   $       0.75      0.60      0.67         5
                   &       0.67      0.67      0.67         6
                   '       0.60      0.33      0.43         9
                   (       0.00      0.00      0.00         1
                   )       0.00      0.00      0.00         1
                   *       0.64      0.64      0.64        14
                   +       0.90      1.00      0.95         9
                   ,       1.00      0.71      0.83         7
                   -       0.69      0.77      0.73       228
                   .       0.76      0.88      0.82       270
                   /       0.77      0.86      0.81       203
                   0       0.89      0.85      0.87        20
                   1       0.94      0.89      0.92   

In [11]:
def predict_next(text, top_k = 3):
    model.eval()
    with torch.no_grad():
        encoded = [vocab.get(t, vocab["<UNK>"]) for t in tokenizer.tokenize(text.lower())]

        encoded = encoded[-context_len:]
        padded = [0] * (context_len - len(encoded)) + encoded
        input_tensor = torch.tensor([padded], dtype=torch.long, device=device)
        probs = torch.softmax(model(input_tensor), dim=1)
        top_probs, top_ids = torch.topk(probs, top_k)
        return [
            (inv_vocab.get(top_ids[0][i].item(), "<UNK>"), top_probs[0][i].item())
            for i in range(top_k)
        ]


text = input("Введите начало команды: ").strip()

predictions = predict_next(text)

print(f"\nВарианты продолжения «{text}»:")
for i, (word, prob) in enumerate(predictions, 1):
    print(f"  {i}. {word}  (уверенность: {prob:.2%})")
print("─" * 30)


Варианты продолжения «sudo systemctl»:
  1. status  (уверенность: 43.18%)
  2. stop  (уверенность: 13.75%)
  3. restart  (уверенность: 12.12%)
──────────────────────────────


In [9]:
torch.save(model.state_dict(), 'metadata/model_weights.pth')
print("Веса сохранены в файл model_weights.pth")

Веса сохранены в файл model_weights.pth


In [10]:
import json

metadata = {
    "vocab": vocab,
    "context_len": context_len,
}

with open("metadata/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)
